In [1]:
import json
import boto3
import os
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
endpoint = os.getenv("S3_ENDPOINT", os.getenv("SEAWEEDFS_S3_ENDPOINT", "http://127.0.0.1:8333"))
if endpoint and not str(endpoint).startswith("http"):
    endpoint = f"http://{endpoint}"

client = boto3.client(
    "s3",
    endpoint_url="http://localhost:7333",
    aws_access_key_id=os.getenv("AWS_ACCESS_KEY_ID", "admin"),
    aws_secret_access_key=os.getenv("AWS_SECRET_ACCESS_KEY", "admin"),
    region_name=os.getenv("AWS_REGION", "us-east-1"),
)
print("S3 endpoint:", endpoint)


S3 endpoint: http://seaweedfs:8333


In [5]:
# Coger el JSON de esta "carpeta" y imprimirlo (si hay varios, usa el primero; cambia a keys[-1] si quieres el último)
bucket = "landing-zone"
prefix = "aemet/08019/2026/05/04/"  # misma ruta que listas arriba; acaba en /

keys = []
for page in client.get_paginator("list_objects_v2").paginate(Bucket=bucket, Prefix=prefix):
    for obj in page.get("Contents", []):
        k = obj["Key"]
        if k.endswith(".json"):
            keys.append(k)

if not keys:
    raise FileNotFoundError(f"No hay ningún .json bajo s3://{bucket}/{prefix}")

key = keys[0]
print("Leyendo:", key, "\n")

raw = client.get_object(Bucket=bucket, Key=key)["Body"].read()
data = json.loads(raw.decode("utf-8"))
print(json.dumps(data, indent=2, ensure_ascii=False))

Leyendo: aemet/08019/2026/05/04/16_hourly_predictions.json 

{
  "id": "08019",
  "nombre": "Barcelona",
  "origen": {
    "web": "https://www.aemet.es",
    "enlace": "https://www.aemet.es/es/eltiempo/prediccion/municipios/horas/barcelona-id08019",
    "language": "es",
    "copyright": "© AEMET. Autorizado el uso de la información y su reproducción citando a AEMET como autora de la misma.",
    "notaLegal": "https://www.aemet.es/es/nota_legal",
    "productor": "Agencia Estatal de Meteorología - AEMET. Gobierno de España"
  },
  "version": "1.0",
  "elaborado": "2026-05-04T16:00:13",
  "provincia": "Barcelona",
  "prediccion": {
    "dia": [
      {
        "orto": "06:44",
        "fecha": "2026-05-04T00:00:00",
        "nieve": [
          {
            "value": "0",
            "periodo": "08"
          },
          {
            "value": "0",
            "periodo": "09"
          },
          {
            "value": "0",
            "periodo": "10"
          },
          {
       

In [6]:
bucket = "landing-zone"
prefix = "aemet/08101/2026/05/04/"  # misma ruta que listas arriba; acaba en /

keys = []
for page in client.get_paginator("list_objects_v2").paginate(Bucket=bucket, Prefix=prefix):
    for obj in page.get("Contents", []):
        k = obj["Key"]
        if k.endswith(".json"):
            keys.append(k)

if not keys:
    raise FileNotFoundError(f"No hay ningún .json bajo s3://{bucket}/{prefix}")

key = keys[0]
print("Leyendo:", key, "\n")

raw = client.get_object(Bucket=bucket, Key=key)["Body"].read()
data = json.loads(raw.decode("utf-8"))
print(json.dumps(data, indent=2, ensure_ascii=False))

Leyendo: aemet/08101/2026/05/04/16_hourly_predictions.json 

{
  "id": "08101",
  "nombre": "Hospitalet de Llobregat, L'",
  "origen": {
    "web": "https://www.aemet.es",
    "enlace": "https://www.aemet.es/es/eltiempo/prediccion/municipios/horas/hospitalet-de-llobregat-l-id08101",
    "language": "es",
    "copyright": "© AEMET. Autorizado el uso de la información y su reproducción citando a AEMET como autora de la misma.",
    "notaLegal": "https://www.aemet.es/es/nota_legal",
    "productor": "Agencia Estatal de Meteorología - AEMET. Gobierno de España"
  },
  "version": "1.0",
  "elaborado": "2026-05-04T16:00:13",
  "provincia": "Barcelona",
  "prediccion": {
    "dia": [
      {
        "orto": "06:44",
        "fecha": "2026-05-04T00:00:00",
        "nieve": [
          {
            "value": "0",
            "periodo": "08"
          },
          {
            "value": "0",
            "periodo": "09"
          },
          {
            "value": "0",
            "periodo": "10

In [7]:
import pyarrow.parquet as pq
import pyarrow as pa
import boto3
import io

client = boto3.client(
    "s3",
    endpoint_url="http://localhost:7333",
    aws_access_key_id="admin",
    aws_secret_access_key="admin",
    region_name="us-east-1",
)

bucket = "persistent-landing-zone"
prefix = "aemet/"

parquets = []
for page in client.get_paginator("list_objects_v2").paginate(Bucket=bucket, Prefix=prefix):
    for obj in page.get("Contents", []):
        if obj["Key"].endswith(".parquet"):
            parquets.append(obj["Key"])

print(f"Parquets trobats: {len(parquets)}")

if parquets:
    raw = client.get_object(Bucket=bucket, Key=parquets[0])["Body"].read()
    
    # Llegeix ignorant els tipus d'extensió de pandas
    reader = pq.ParquetFile(io.BytesIO(raw))
    table = reader.read()
    
    # Converteix a pandas ignorant metadades problemàtiques
    df = table.to_pandas(timestamp_as_object=True)
    print(f"Total files: {len(df)}")
    print(df.columns)
    print(df['prediccion'])
    print(df.head())
    print(df.dtypes)

    print(df.isna().sum())

Parquets trobats: 2
Total files: 2
Index(['id', 'nombre', 'origen', 'version', 'elaborado', 'provincia',
       'prediccion'],
      dtype='str')
0    {'dia': [{'orto': '06:44', 'fecha': '2026-05-0...
1    {'dia': [{'orto': '06:44', 'fecha': '2026-05-0...
Name: prediccion, dtype: str
      id                       nombre  \
0  08019                    Barcelona   
1  08101  Hospitalet de Llobregat, L'   

                                              origen version  \
0  {'web': 'https://www.aemet.es', 'enlace': 'htt...     1.0   
1  {'web': 'https://www.aemet.es', 'enlace': 'htt...     1.0   

             elaborado  provincia  \
0  2026-05-04T16:00:13  Barcelona   
1  2026-05-04T16:00:13  Barcelona   

                                          prediccion  
0  {'dia': [{'orto': '06:44', 'fecha': '2026-05-0...  
1  {'dia': [{'orto': '06:44', 'fecha': '2026-05-0...  
id            str
nombre        str
origen        str
version       str
elaborado     str
provincia     str
prediccion   

In [28]:
import pyarrow.parquet as pq
import pyarrow as pa
import boto3
import io

client = boto3.client(
    "s3",
    endpoint_url="http://localhost:7333",
    aws_access_key_id="admin",
    aws_secret_access_key="admin",
    region_name="us-east-1",
)

bucket = "trusted-zone"
prefix = "aemet/"

parquets = []
for page in client.get_paginator("list_objects_v2").paginate(Bucket=bucket, Prefix=prefix):
    for obj in page.get("Contents", []):
        if obj["Key"].endswith(".parquet"):
            parquets.append(obj["Key"])

print(f"Parquets trobats: {len(parquets)}")

if parquets:
    raw = client.get_object(Bucket=bucket, Key=parquets[0])["Body"].read()
    
    # Llegeix ignorant els tipus d'extensió de pandas
    reader = pq.ParquetFile(io.BytesIO(raw))
    table = reader.read()
    
    # Converteix a pandas ignorant metadades problemàtiques
    df = table.to_pandas(timestamp_as_object=True)
    print(f"Total files: {len(df)}")
    print(df.columns)
    print(df['prediccion'])
    print(df.head())
    print(df.dtypes)
    print(df.isna().sum())

Parquets trobats: 1
Total files: 2
Index(['id', 'nombre', 'origen', 'version', 'elaborado', 'provincia',
       'prediccion'],
      dtype='str')
0    {'dia': [{'orto': '06:46', 'fecha': '2026-05-0...
1    {'dia': [{'orto': '06:45', 'fecha': '2026-05-0...
Name: prediccion, dtype: str
      id                       nombre  \
0  08101  Hospitalet de Llobregat, L'   
1  08019                    Barcelona   

                                              origen version  \
0  {'web': 'https://www.aemet.es', 'enlace': 'htt...     1.0   
1  {'web': 'https://www.aemet.es', 'enlace': 'htt...     1.0   

             elaborado  provincia  \
0  2026-05-03T18:52:12  Barcelona   
1  2026-05-03T18:52:12  Barcelona   

                                          prediccion  
0  {'dia': [{'orto': '06:46', 'fecha': '2026-05-0...  
1  {'dia': [{'orto': '06:45', 'fecha': '2026-05-0...  
id            str
nombre        str
origen        str
version       str
elaborado     str
provincia     str
prediccion   

In [ ]:
import pyarrow.parquet as pq
import pyarrow as pa
import boto3
import io

client = boto3.client(
    "s3",
    endpoint_url="http://localhost:7333",
    aws_access_key_id="admin",
    aws_secret_access_key="admin",
    region_name="us-east-1",
)

bucket = "exploitation-zone"
prefix = "aemet/"

parquets = []
for page in client.get_paginator("list_objects_v2").paginate(Bucket=bucket, Prefix=prefix):
    for obj in page.get("Contents", []):
        if obj["Key"].endswith(".parquet"):
            parquets.append(obj["Key"])

print(f"Parquets trobats: {len(parquets)}")

if parquets:
    raw = client.get_object(Bucket=bucket, Key=parquets[0])["Body"].read()
    
    # Llegeix ignorant els tipus d'extensió de pandas
    reader = pq.ParquetFile(io.BytesIO(raw))
    table = reader.read()
    
    # Converteix a pandas ignorant metadades problemàtiques
    df = table.to_pandas(timestamp_as_object=True)
    print(f"Total files: {len(df)}")
    print(df.columns)
    print(df['prediccion'])
    print(df.head())
    print(df.dtypes)
    

Parquets trobats: 1
Total files: 2
Index(['id', 'nombre', 'origen', 'version', 'elaborado', 'provincia',
       'prediccion'],
      dtype='str')
0    {'dia': [{'orto': '06:46', 'fecha': '2026-05-0...
1    {'dia': [{'orto': '06:45', 'fecha': '2026-05-0...
Name: prediccion, dtype: str
      id                       nombre  \
0  08101  Hospitalet de Llobregat, L'   
1  08019                    Barcelona   

                                              origen version  \
0  {'web': 'https://www.aemet.es', 'enlace': 'htt...     1.0   
1  {'web': 'https://www.aemet.es', 'enlace': 'htt...     1.0   

             elaborado  provincia  \
0  2026-05-03T18:52:12  Barcelona   
1  2026-05-03T18:52:12  Barcelona   

                                          prediccion  
0  {'dia': [{'orto': '06:46', 'fecha': '2026-05-0...  
1  {'dia': [{'orto': '06:45', 'fecha': '2026-05-0...  
id            str
nombre        str
origen        str
version       str
elaborado     str
provincia     str
prediccion   

**AJUNTAMENT DE BARCELONA**

*ACCIDENTS_GU*

In [8]:
import pandas as pd
import boto3
import io

client = boto3.client(
    "s3",
    endpoint_url="http://localhost:7333",
    aws_access_key_id="admin",
    aws_secret_access_key="admin",
    region_name="us-east-1",
)

bucket = "landing-zone"
prefix = "ajuntament_bcn/accidents_gu/"

# Llista tots els CSVs
csvs = []
for page in client.get_paginator("list_objects_v2").paginate(Bucket=bucket, Prefix=prefix):
    for obj in page.get("Contents", []):
        if obj["Key"].endswith(".csv"):
            csvs.append(obj["Key"])

print(f"CSVs trobats: {len(csvs)}")

# Llegeix tots i ajunta'ls en un sol DataFrame
dfs = []
for key in csvs:
    raw = client.get_object(Bucket=bucket, Key=key)["Body"].read()
    df = pd.read_csv(io.BytesIO(raw))
    print(f"  → {key}: {len(df)} files -> columns: {df.columns} -> numero de nulos: {df.isna().sum()}")
    print(df['NK_Any'])
    dfs.append(df)

# Ajunta tots
df_total = pd.concat(dfs, ignore_index=True)
print(f"\nTotal files combinades: {len(df_total)}")
print(df_total)


CSVs trobats: 5
  → ajuntament_bcn/accidents_gu/2021_ACCIDENTS_GU_BCN.csv: 7659 files -> columns: Index(['Numero_expedient', 'Codi_districte', 'Nom_districte', 'Codi_barri',
       'Nom_barri', 'Codi_carrer', 'Nom_carrer', 'Num_postal_caption',
       'Descripcio_dia_setmana', 'NK_Any', 'Mes_any', 'Nom_mes', 'Dia_mes',
       'Hora_dia', 'Descripcio_torn', 'Descripcio_causa_vianant',
       'Numero_morts', 'Numero_lesionats_lleus', 'Numero_lesionats_greus',
       'Numero_victimes', 'Numero_vehicles_implicats', 'Coordenada_UTM_X_ED50',
       'Coordenada_UTM_Y_ED50', 'Longitud_WGS84', 'Latitud_WGS84'],
      dtype='str') -> numero de nulos: Numero_expedient              0
Codi_districte                0
Nom_districte                 0
Codi_barri                    0
Nom_barri                     0
Codi_carrer                   0
Nom_carrer                    0
Num_postal_caption            0
Descripcio_dia_setmana        0
NK_Any                        0
Mes_any                       0

In [9]:
import pyarrow.parquet as pq
import pyarrow as pa
import boto3
import io

client = boto3.client(
    "s3",
    endpoint_url="http://localhost:7333",
    aws_access_key_id="admin",
    aws_secret_access_key="admin",
    region_name="us-east-1",
)

bucket = "persistent-landing-zone"
prefix = "ajuntament_bcn/accidents_gu/"

parquets = []
for page in client.get_paginator("list_objects_v2").paginate(Bucket=bucket, Prefix=prefix):
    for obj in page.get("Contents", []):
        if obj["Key"].endswith(".parquet"):
            parquets.append(obj["Key"])

print(f"Parquets trobats: {len(parquets)}")

if parquets:
    raw = client.get_object(Bucket=bucket, Key=parquets[0])["Body"].read()
    
    # Llegeix ignorant els tipus d'extensió de pandas
    reader = pq.ParquetFile(io.BytesIO(raw))
    table = reader.read()
    
    # Converteix a pandas ignorant metadades problemàtiques
    df = table.to_pandas(timestamp_as_object=True)
    print(f"Total files: {len(df)}")
    print(df.columns)
    print(df.head())
    print(df['NK_Any'])
    print(df.dtypes)
    

Parquets trobats: 1
Total files: 7741
Index(['ï»¿Numero_expedient', 'Codi_districte', 'Nom_districte', 'Codi_barri',
       'Nom_barri', 'Codi_carrer', 'Nom_carrer', 'Num_postal ',
       'Descripcio_dia_setmana', 'NK_Any', 'Mes_any', 'Nom_mes', 'Dia_mes',
       'Hora_dia', 'Descripcio_torn', 'Descripcio_causa_vianant',
       'Numero_morts', 'Numero_lesionats_lleus', 'Numero_lesionats_greus',
       'Numero_victimes', 'Numero_vehicles_implicats', 'Coordenada_UTM_Y_ED50',
       'Coordenada_UTM_X_ED50', 'Longitud_WGS84', 'Latitud_WGS84'],
      dtype='str')
  ï»¿Numero_expedient  Codi_districte         Nom_districte  Codi_barri  \
0         2025S000001               3       Sants-MontjuÃ¯c          11   
1         2025S000002              10           Sant MartÃ­          65   
2         2025S000003              10           Sant MartÃ­          66   
3         2025S000004               5  SarriÃ -Sant Gervasi          26   
4         2025S000005               4             Les Corts 

In [41]:
import pyarrow.parquet as pq
import pyarrow as pa
import boto3
import io

client = boto3.client(
    "s3",
    endpoint_url="http://localhost:7333",
    aws_access_key_id="admin",
    aws_secret_access_key="admin",
    region_name="us-east-1",
)

bucket = "exploitation-zone"
prefix = "ajuntament_bcn/accidents_gu/"

parquets = []
for page in client.get_paginator("list_objects_v2").paginate(Bucket=bucket, Prefix=prefix):
    for obj in page.get("Contents", []):
        if obj["Key"].endswith(".parquet"):
            parquets.append(obj["Key"])

print(f"Parquets trobats: {len(parquets)}")

if parquets:
    raw = client.get_object(Bucket=bucket, Key=parquets[0])["Body"].read()
    
    # Llegeix ignorant els tipus d'extensió de pandas
    reader = pq.ParquetFile(io.BytesIO(raw))
    table = reader.read()
    
    # Converteix a pandas ignorant metadades problemàtiques
    df = table.to_pandas(timestamp_as_object=True)
    print(f"Total files: {len(df)}")
    print(df.columns)
    print(df.head())
    print(df['NK_Any'])
    print(df.dtypes)
    

Parquets trobats: 2
Total files: 19401
Index(['Numero_expedient', 'Codi_districte', 'Nom_districte', 'Codi_barri',
       'Nom_barri', 'Codi_carrer', 'Nom_carrer', 'Num_postal_caption',
       'Descripcio_dia_setmana', 'NK_Any', 'Mes_any', 'Nom_mes', 'Dia_mes',
       'Hora_dia', 'Descripcio_torn', 'Descripcio_causa_vianant',
       'Numero_morts', 'Numero_lesionats_lleus', 'Numero_lesionats_greus',
       'Numero_victimes', 'Numero_vehicles_implicats', 'Coordenada_UTM_X_ED50',
       'Coordenada_UTM_Y_ED50', 'Longitud_WGS84', 'Latitud_WGS84', 'Longitud',
       'Latitud', 'Num_postal ', 'ï»¿Numero_expedient'],
      dtype='str')
  Numero_expedient  Codi_districte    Nom_districte  Codi_barri  \
0      2021S003741               7  Horta-GuinardÃ³          41   
1      2021S004338               7  Horta-GuinardÃ³          41   
2      2021S000830               1     Ciutat Vella           1   
3      2021S003342               4        Les Corts          20   
4      2023S000058         

*TRAMS*

In [43]:
import pandas as pd
import boto3
import io

client = boto3.client(
    "s3",
    endpoint_url="http://localhost:7333",
    aws_access_key_id="admin",
    aws_secret_access_key="admin",
    region_name="us-east-1",
)

bucket = "landing-zone"
prefix = "ajuntament_bcn/trams_congestion/"

# Llista tots els CSVs
csvs = []
for page in client.get_paginator("list_objects_v2").paginate(Bucket=bucket, Prefix=prefix):
    for obj in page.get("Contents", []):
        if obj["Key"].endswith(".csv"):
            csvs.append(obj["Key"])

print(f"CSVs trobats: {len(csvs)}")

# Llegeix tots i ajunta'ls en un sol DataFrame
dfs = []
for key in csvs:
    raw = client.get_object(Bucket=bucket, Key=key)["Body"].read()
    df = pd.read_csv(io.BytesIO(raw))
    print(f"  → {key}: {len(df)} files -> columns: {df.columns} -> numero de nulos: {df.isna().sum()}")
    dfs.append(df)

# Ajunta tots
df_total = pd.concat(dfs, ignore_index=True)
print(f"\nTotal files combinades: {len(df_total)}")
print(df_total)


CSVs trobats: 13
  → ajuntament_bcn/trams_congestion/2025_01_Gener_TRAMS_TRAMS.csv: 1970528 files -> columns: Index(['idTram', 'data', 'estatActual', 'estatPrevist'], dtype='str') -> numero de nulos: idTram          0
data            0
estatActual     0
estatPrevist    0
dtype: int64
  → ajuntament_bcn/trams_congestion/2025_02_Febrer_TRAMS_TRAMS.csv: 1786456 files -> columns: Index(['idTram', 'data', 'estatActual', 'estatPrevist'], dtype='str') -> numero de nulos: idTram          0
data            0
estatActual     0
estatPrevist    0
dtype: int64
  → ajuntament_bcn/trams_congestion/2025_03_Marc_TRAMS_TRAMS.csv: 2207800 files -> columns: Index(['idTram', 'data', 'estatActual', 'estatPrevist'], dtype='str') -> numero de nulos: idTram          0
data            0
estatActual     0
estatPrevist    0
dtype: int64
  → ajuntament_bcn/trams_congestion/2025_04_Abril_TRAMS_TRAMS.csv: 1542800 files -> columns: Index(['idTram', 'data', 'estatActual', 'estatPrevist'], dtype='str') -> numero de nul

In [45]:
import pyarrow.parquet as pq
import pyarrow as pa
import boto3
import io

client = boto3.client(
    "s3",
    endpoint_url="http://localhost:7333",
    aws_access_key_id="admin",
    aws_secret_access_key="admin",
    region_name="us-east-1",
)

bucket = "persistent-landing-zone"
prefix = "ajuntament_bcn/trams_congestion/"

parquets = []
for page in client.get_paginator("list_objects_v2").paginate(Bucket=bucket, Prefix=prefix):
    for obj in page.get("Contents", []):
        if obj["Key"].endswith(".parquet"):
            parquets.append(obj["Key"])

print(f"Parquets trobats: {len(parquets)}")

if parquets:
    raw = client.get_object(Bucket=bucket, Key=parquets[0])["Body"].read()
    
    # Llegeix ignorant els tipus d'extensió de pandas
    reader = pq.ParquetFile(io.BytesIO(raw))
    table = reader.read()
    
    # Converteix a pandas ignorant metadades problemàtiques
    df = table.to_pandas(timestamp_as_object=True)
    print(f"Total files: {len(df)}")
    print(df.columns)
    print(df.head())
    print(df.dtypes)
    

Parquets trobats: 2
Total files: 531
Index(['1#20260503220054#2#2'], dtype='str')
   1#20260503220054#2#2
0  2#20260503220054#1#1
1  3#20260503220054#2#2
2  4#20260503220054#2#2
3  5#20260503220054#2#2
4  6#20260503220054#2#2
1#20260503220054#2#2    str
dtype: object


In [46]:
import pyarrow.parquet as pq
import pyarrow as pa
import boto3
import io

client = boto3.client(
    "s3",
    endpoint_url="http://localhost:7333",
    aws_access_key_id="admin",
    aws_secret_access_key="admin",
    region_name="us-east-1",
)

bucket = "exploitation-zone"
prefix = "ajuntament_bcn/trams_congestion/"

parquets = []
for page in client.get_paginator("list_objects_v2").paginate(Bucket=bucket, Prefix=prefix):
    for obj in page.get("Contents", []):
        if obj["Key"].endswith(".parquet"):
            parquets.append(obj["Key"])

print(f"Parquets trobats: {len(parquets)}")

if parquets:
    raw = client.get_object(Bucket=bucket, Key=parquets[0])["Body"].read()
    
    # Llegeix ignorant els tipus d'extensió de pandas
    reader = pq.ParquetFile(io.BytesIO(raw))
    table = reader.read()
    
    # Converteix a pandas ignorant metadades problemàtiques
    df = table.to_pandas(timestamp_as_object=True)
    print(f"Total files: {len(df)}")
    print(df.columns)
    print(df.head())
    print(df.dtypes)
    

Parquets trobats: 1
Total files: 531
Index(['1#20260503220054#2#2'], dtype='str')
     1#20260503220054#2#2
0   10#20260503220054#1#1
1  115#20260503220054#1#1
2  464#20260503220054#0#0
3  294#20260503220054#0#0
4  455#20260503220054#0#0
1#20260503220054#2#2    str
dtype: object


**SCT TRAFFICI INCIDENTS**

In [47]:
import boto3

client = boto3.client(
    "s3",
    endpoint_url="http://localhost:7333",
    aws_access_key_id="admin",
    aws_secret_access_key="admin",
    region_name="us-east-1",
)

bucket = "landing-zone"
prefix = "sct/traffic_incidents/"

# Llista tots els XMLs
xmls = []
for page in client.get_paginator("list_objects_v2").paginate(Bucket=bucket, Prefix=prefix):
    for obj in page.get("Contents", []):
        if obj["Key"].endswith(".xml"):
            xmls.append(obj["Key"])

print(f"XMLs trobats: {len(xmls)}")

# Llegeix i imprimeix cada un
for key in xmls:
    print(f"\n{'='*50}")
    print(f"Fitxer: {key}")
    print('='*50)
    raw = client.get_object(Bucket=bucket, Key=key)["Body"].read()
    print(raw.decode("utf-8"))

XMLs trobats: 2

Fitxer: sct/traffic_incidents/2026-05-03T20:05:00+00:00.xml
<?xml version="1.0" encoding="UTF-8"?><wfs:FeatureCollection xmlns="http://www.opengis.net/wfs" xmlns:wfs="http://www.opengis.net/wfs" xmlns:gml="http://www.opengis.net/gml" xmlns:cite="http://www.opengeospatial.net/cite" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance" xsi:schemaLocation="http://www.opengeospatial.net/cite http://localhost:8080/sct-gis/wfs?service=WFS&amp;version=1.0.0&amp;request=DescribeFeatureType&amp;typeName=cite%3Amct2_v_afectacions_data http://www.opengis.net/wfs http://localhost:8080/sct-gis/schemas/wfs/1.0.0/WFS-basic.xsd"><gml:boundedBy><gml:null>unknown</gml:null></gml:boundedBy><gml:featureMember><cite:mct2_v_afectacions_data fid="mct2_v_afectacions_data.fid-1f259c63_19dee4570b0_4b6"><cite:geom><gml:Point srsName="http://www.opengis.net/gml/srs/epsg.xml#4326"><gml:coordinates xmlns:gml="http://www.opengis.net/gml" decimal="." cs="," ts=" ">2.26631924,41.63483065</gml:coordina

In [49]:
import pyarrow.parquet as pq
import pyarrow as pa
import boto3
import io

client = boto3.client(
    "s3",
    endpoint_url="http://localhost:7333",
    aws_access_key_id="admin",
    aws_secret_access_key="admin",
    region_name="us-east-1",
)

bucket = "exploitation-zone"
prefix = "sct/traffic_incidents/"

parquets = []
for page in client.get_paginator("list_objects_v2").paginate(Bucket=bucket, Prefix=prefix):
    for obj in page.get("Contents", []):
        if obj["Key"].endswith(".parquet"):
            parquets.append(obj["Key"])

print(f"Parquets trobats: {len(parquets)}")

if parquets:
    raw = client.get_object(Bucket=bucket, Key=parquets[0])["Body"].read()
    
    # Llegeix ignorant els tipus d'extensió de pandas
    reader = pq.ParquetFile(io.BytesIO(raw))
    table = reader.read()
    
    # Converteix a pandas ignorant metadades problemàtiques
    df = table.to_pandas(timestamp_as_object=True)
    print(f"Total files: {len(df)}")
    print(df.columns)
    print(df.head())
    print(df.columns)
    print(df.dtypes)
    

Parquets trobats: 1
Total files: 2
Index(['file_name', 'xml_content'], dtype='str')
                                           file_name  \
0  sct/traffic_incidents/2026-05-03T20:10:00+00:0...   
1  sct/traffic_incidents/2026-05-03T20:05:00+00:0...   

                                         xml_content  
0  <?xml version="1.0" encoding="UTF-8"?><wfs:Fea...  
1  <?xml version="1.0" encoding="UTF-8"?><wfs:Fea...  
Index(['file_name', 'xml_content'], dtype='str')
file_name      str
xml_content    str
dtype: object


In [50]:
bucket = "landing-zone"
prefix = "transit_posts"  # misma ruta que listas arriba; acaba en /

keys = []
for page in client.get_paginator("list_objects_v2").paginate(Bucket=bucket, Prefix=prefix):
    for obj in page.get("Contents", []):
        k = obj["Key"]
        if k.endswith(".json"):
            keys.append(k)

if not keys:
    raise FileNotFoundError(f"No hay ningún .json bajo s3://{bucket}/{prefix}")

key = keys[0]
print("Leyendo:", key, "\n")

raw = client.get_object(Bucket=bucket, Key=key)["Body"].read()
data = json.loads(raw.decode("utf-8"))
print(json.dumps(data, indent=2, ensure_ascii=False))

Leyendo: transit_posts/post_2026-05-03T15:27:24+00:00.json 

{
  "title": "🔴❌ A la C-58 hi ha un carril tallat a Montcada en sentit Sabadell per un vehicle #avariat. Retencions des del Nus de la Trinitat. També afecta la ronda Litoral des de Sant Adrià de Besòs. \n\n🔴 A la C-58 i a la C-33 també hi ha cua de Montcada a Barcelona en sentit sud.",
  "title_detail": {
    "type": "text/plain",
    "language": null,
    "base": "https://nitter.net/transit/rss",
    "value": "🔴❌ A la C-58 hi ha un carril tallat a Montcada en sentit Sabadell per un vehicle #avariat. Retencions des del Nus de la Trinitat. També afecta la ronda Litoral des de Sant Adrià de Besòs. \n\n🔴 A la C-58 i a la C-33 també hi ha cua de Montcada a Barcelona en sentit sud."
  },
  "authors": [
    {
      "name": "@transit"
    }
  ],
  "author": "@transit",
  "author_detail": {
    "name": "@transit"
  },
  "summary": "<p>🔴❌ A la C-58 hi ha un carril tallat a Montcada en sentit Sabadell per un vehicle <a href=\"https://n

In [52]:
import pyarrow.parquet as pq
import pyarrow as pa
import boto3
import io

client = boto3.client(
    "s3",
    endpoint_url="http://localhost:7333",
    aws_access_key_id="admin",
    aws_secret_access_key="admin",
    region_name="us-east-1",
)

bucket = "persistent-landing-zone"
prefix = "transit_posts"

parquets = []
for page in client.get_paginator("list_objects_v2").paginate(Bucket=bucket, Prefix=prefix):
    for obj in page.get("Contents", []):
        if obj["Key"].endswith(".parquet"):
            parquets.append(obj["Key"])

print(f"Parquets trobats: {len(parquets)}")

if parquets:
    raw = client.get_object(Bucket=bucket, Key=parquets[0])["Body"].read()
    
    # Llegeix ignorant els tipus d'extensió de pandas
    reader = pq.ParquetFile(io.BytesIO(raw))
    table = reader.read()
    
    # Converteix a pandas ignorant metadades problemàtiques
    df = table.to_pandas(timestamp_as_object=True)
    print(f"Total files: {len(df)}")
    print(df.columns)
    print(df.head())
    print(df.columns)
    print(df.dtypes)
    

Parquets trobats: 2
Total files: 21
Index(['title', 'title_detail', 'authors', 'author', 'author_detail',
       'summary', 'summary_detail', 'published', 'published_parsed', 'id',
       'guidislink', 'links', 'link'],
      dtype='str')
                                               title  \
0  ACCIDENT. Calçada tallada (Retenció): C-17 | C...   
1  🔴❌ A la C-58 hi ha un carril tallat a Montcada...   
2  🔴 A la ronda Litoral de Barcelona hi ha retenc...   
3  R to @transit: 🟢 Situació normalitzada a l'AP-...   
4  R to @transit: 🔴 A l'A-2 ara hi ha lentitud i ...   

                                        title_detail                 authors  \
0  {'type': 'text/plain', 'language': None, 'base...  [{'name': '@transit'}]   
1  {'type': 'text/plain', 'language': None, 'base...  [{'name': '@transit'}]   
2  {'type': 'text/plain', 'language': None, 'base...  [{'name': '@transit'}]   
3  {'type': 'text/plain', 'language': None, 'base...  [{'name': '@transit'}]   
4  {'type': 'text/plain'

In [53]:
import pyarrow.parquet as pq
import pyarrow as pa
import boto3
import io

client = boto3.client(
    "s3",
    endpoint_url="http://localhost:7333",
    aws_access_key_id="admin",
    aws_secret_access_key="admin",
    region_name="us-east-1",
)

bucket = "exploitation-zone"
prefix = "transit_posts"

parquets = []
for page in client.get_paginator("list_objects_v2").paginate(Bucket=bucket, Prefix=prefix):
    for obj in page.get("Contents", []):
        if obj["Key"].endswith(".parquet"):
            parquets.append(obj["Key"])

print(f"Parquets trobats: {len(parquets)}")

if parquets:
    raw = client.get_object(Bucket=bucket, Key=parquets[0])["Body"].read()
    
    # Llegeix ignorant els tipus d'extensió de pandas
    reader = pq.ParquetFile(io.BytesIO(raw))
    table = reader.read()
    
    # Converteix a pandas ignorant metadades problemàtiques
    df = table.to_pandas(timestamp_as_object=True)
    print(f"Total files: {len(df)}")
    print(df.columns)
    print(df.head())
    print(df.columns)
    print(df.dtypes)
    

Parquets trobats: 1
Total files: 21
Index(['title', 'title_detail', 'authors', 'author', 'author_detail',
       'summary', 'summary_detail', 'published', 'published_parsed', 'id',
       'guidislink', 'links', 'link'],
      dtype='str')
                                               title  \
0  ℹ  Oberts també els carrils addicionals en sen...   
1  R to @transit: 🔴 A la C-32 hi ha retencions en...   
2  R to @transit: ⚫ L'A-2 ara està tallada a Abre...   
3  🔴 A la ronda Litoral de Barcelona hi ha retenc...   
4  R to @transit: 🟢 Situació normalitzada a l'AP-...   

                                        title_detail                 authors  \
0  {'type': 'text/plain', 'language': None, 'base...  [{'name': '@transit'}]   
1  {'type': 'text/plain', 'language': None, 'base...  [{'name': '@transit'}]   
2  {'type': 'text/plain', 'language': None, 'base...  [{'name': '@transit'}]   
3  {'type': 'text/plain', 'language': None, 'base...  [{'name': '@transit'}]   
4  {'type': 'text/plain'

**CANDIDATS A FER JOIN**

ELS QUE JO FARIA JOIN SI O SI, SERIEN ELS SEGUENTS:

*1. Relacio trams amb Trams congestion.*

*2.EL QUE FARIA AMB ACCIDENTS_GU, ES UN MODEL PREDICTIU QUE DESPRÉS PRINTES RESULTATS AL DASHBOARD*